# DINOv2 Embedding POC — Before Fine-Tuning

**Goal:** test whether pretrained **DINOv2 embeddings** contain useful patterns for the wheat dataset **before any fine-tuning**.

This notebook does **not** train a classifier by default. It:

1. Loads the labeled wheat dataset.
2. Extracts individual grains from YOLO segmentation/detection labels when available.
3. Extracts frozen DINOv2 embeddings.
4. Saves the embeddings + metadata.
5. Checks class separability, redundancy, and embedding consistency.
6. Optionally runs a simple linear probe only as a sanity check.

> Baseline rule: start with the original images and DINOv2's required preprocessing only. Do not apply color normalization before establishing the baseline.


In [1]:
# Print Embedding  C:\Users\alool\Desktop\Codes\Python\Test_RFDeter\notebooks_Test_DinoV2\DINOv2_Comprehensive_Embedding_Evaluation.html

import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path(
    r"C:\Users\alool\Desktop\Codes\Python\Test_RFDeter"
    r"\notebooks_Test_DinoV2\dinov2_comprehensive_eval"
)

embedding_file = (
    BASE_DIR
    / "embedding_cache"
    / "vit_s14__original__clean__aspect_pad.npz"
)

metadata_file = (
    BASE_DIR
    / "results"
    / "representative_subset_manifest.csv"
)

output_file = (
    BASE_DIR
    / "results"
    / "best_cls_embeddings_with_metadata.csv"
)

# 1. Read the best CLS embeddings
with np.load(embedding_file, allow_pickle=False) as data:
    X = data["cls"].copy()
    row_ids = data["row_ids"].copy()

print("Embedding matrix:", X.shape)

# 2. Read and align metadata with the embedding rows
metadata = pd.read_csv(metadata_file)

metadata = (
    metadata
    .set_index("row_id")
    .loc[row_ids]
    .reset_index()
)

# 3. Give every embedding dimension a clear column name
embedding_columns = [
    f"cls_feature_{i:03d}"
    for i in range(X.shape[1])
]

embeddings_df = pd.DataFrame(
    X,
    columns=embedding_columns
)

# 4. Add information describing the selected configuration
configuration_df = pd.DataFrame({
    "model": ["dinov2_vits14"] * len(metadata),
    "normalization": ["original"] * len(metadata),
    "resize_mode": ["aspect_pad"] * len(metadata),
    "embedding_type": ["cls"] * len(metadata),
})

# 5. Combine metadata and all 384 embedding values
final_df = pd.concat(
    [
        configuration_df.reset_index(drop=True),
        metadata.reset_index(drop=True),
        embeddings_df.reset_index(drop=True),
    ],
    axis=1
)

# 6. Save to CSV
# 9 significant digits are sufficient to preserve float32 values.
final_df.to_csv(
    output_file,
    index=False,
    float_format="%.9g",
)

print("Saved successfully:", output_file)
print("CSV shape:", final_df.shape)
print("Number of grains:", len(final_df))
print("Embedding columns:", X.shape[1])

display(final_df.head())

Embedding matrix: (3000, 384)
Saved successfully: C:\Users\alool\Desktop\Codes\Python\Test_RFDeter\notebooks_Test_DinoV2\dinov2_comprehensive_eval\results\best_cls_embeddings_with_metadata.csv
CSV shape: (3000, 396)
Number of grains: 3000
Embedding columns: 384


,model,normalization,resize_mode,embedding_type,row_id,split,image_path,label_path,instance_idx,class_id,...,cls_feature_374,cls_feature_375,cls_feature_376,cls_feature_377,cls_feature_378,cls_feature_379,cls_feature_380,cls_feature_381,cls_feature_382,cls_feature_383
0,dinov2_vits14,original,aspect_pad,cls,0,train,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,48,1,...,0.983076,7.332440,0.182072,-0.224813,-3.397323,-1.205513,0.640060,-1.457910,1.607962,-1.953028
1,dinov2_vits14,original,aspect_pad,cls,1,train,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,1,1,...,0.280234,5.571773,2.357996,-4.135096,-0.937437,-1.186574,-0.237129,1.332036,1.712007,-0.797890
2,dinov2_vits14,original,aspect_pad,cls,2,train,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,168,1,...,0.997305,5.153206,0.161891,0.490629,-0.139083,-0.305216,-1.187576,-0.017364,-3.006669,-1.621859
3,dinov2_vits14,original,aspect_pad,cls,3,train,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,43,0,...,-0.495402,4.883095,2.194621,-0.531830,-1.076792,-0.384075,-1.201207,0.645074,2.724857,-0.023033
4,dinov2_vits14,original,aspect_pad,cls,4,train,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,C:\Users\alool\Desktop\Codes\Python\Test_RFDet...,1,2,...,-1.318904,6.691782,-1.220461,3.979213,-1.946602,-0.080175,0.088046,2.836971,1.319552,-1.238513


In [ ]:
# Run this once if the environment does not already contain the packages.
# In Jupyter, uncomment the next line and run the cell.

# %pip install -U torch torchvision transformers scikit-learn pandas numpy pillow matplotlib tqdm pyyaml umap-learn


## 1. Configuration

In [ ]:
from pathlib import Path

# Your paths
DATASET_DIR = Path(r"C:\Users\alool\Desktop\Codes\Python\Test_RFDeter\wheatdecv1_seg")
OUTPUT_DIR = Path(r"C:\Users\alool\Desktop\Codes\Python\Test_RFDeter\notebooks_Test_DinoV2\dinov2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# DINOv2 model
# Use "facebook/dinov2-small" for a faster/light test.
# Use "facebook/dinov2-base" for a stronger baseline.
MODEL_NAME = "facebook/dinov2-base"

# Embedding pooling:
# "cls"      -> CLS token (recommended baseline)
# "mean"     -> mean of patch tokens
# "cls_mean" -> concatenate CLS + mean patch token
POOLING = "cls"

# Dataset mode:
# "auto" -> automatically detect YOLO segmentation/detection or folder-classification format
# "yolo" -> force YOLO-style labels
# "classification" -> folders/classes with one image per grain
DATASET_MODE = "auto"

# Limits
MAX_SAMPLES = None        # Example: 500 for a quick test, None = all samples/instances
SAVE_GRAIN_CROPS = True   # Useful to visually verify what DINOv2 actually receives
MASK_BACKGROUND = True    # If polygon masks exist, remove neighboring grain/background pixels
CROP_PADDING = 4          # Pixels around each extracted grain

# Performance
BATCH_SIZE = 32
NUM_WORKERS = 0           # Safer on Windows/Jupyter
RANDOM_SEED = 42

# Optional analyses
RUN_UMAP = True
RUN_LINEAR_PROBE = False  # Keep False for pure embedding analysis


## 2. Imports and device check

In [ ]:
import os
import math
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance
from tqdm.auto import tqdm
import yaml

from transformers import AutoImageProcessor, AutoModel
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Dataset exists:", DATASET_DIR.exists(), DATASET_DIR)
print("Output directory:", OUTPUT_DIR)


## 3. Dataset loader

The notebook supports two common cases:

### A) YOLO segmentation/detection dataset
Typical structure:

```text
wheatdecv1_seg/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

For segmentation labels, each polygon instance becomes one grain sample for DINOv2.

### B) Classification folders
If every image already contains one isolated grain:

```text
dataset/
├── healthy/
├── bad/
└── broken/
```


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def load_class_names(dataset_dir: Path):
    yaml_candidates = list(dataset_dir.glob("*.yaml")) + list(dataset_dir.glob("*.yml"))
    if not yaml_candidates:
        return {}
    with open(yaml_candidates[0], "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    names = cfg.get("names", {})
    if isinstance(names, list):
        return {i: str(name) for i, name in enumerate(names)}
    if isinstance(names, dict):
        return {int(k): str(v) for k, v in names.items()}
    return {}

CLASS_NAMES = load_class_names(DATASET_DIR)
print("Class names from YAML:", CLASS_NAMES)


In [ ]:
def find_yolo_pairs(dataset_dir: Path):
    pairs = []
    split_aliases = ["train", "valid", "val", "test"]

    for split in split_aliases:
        # Layout 1: split/images + split/labels
        img_dir = dataset_dir / split / "images"
        lbl_dir = dataset_dir / split / "labels"

        # Layout 2: images/split + labels/split
        if not img_dir.exists():
            img_dir = dataset_dir / "images" / split
            lbl_dir = dataset_dir / "labels" / split

        if img_dir.exists() and lbl_dir.exists():
            normalized_split = "valid" if split == "val" else split
            for img_path in img_dir.rglob("*"):
                if img_path.suffix.lower() not in IMAGE_EXTS:
                    continue
                rel = img_path.relative_to(img_dir).with_suffix(".txt")
                label_path = lbl_dir / rel
                if label_path.exists():
                    pairs.append((normalized_split, img_path, label_path))

    return pairs


def detect_dataset_mode(dataset_dir: Path):
    if DATASET_MODE != "auto":
        return DATASET_MODE

    yolo_pairs = find_yolo_pairs(dataset_dir)
    if yolo_pairs:
        return "yolo"

    # Classification fallback: image files inside class folders
    class_dirs = [
        p for p in dataset_dir.iterdir()
        if p.is_dir() and any(x.suffix.lower() in IMAGE_EXTS for x in p.rglob("*"))
    ]
    if class_dirs:
        return "classification"

    raise RuntimeError(
        "Could not auto-detect the dataset format. "
        "Set DATASET_MODE manually after checking the folder structure."
    )

MODE = detect_dataset_mode(DATASET_DIR)
print("Detected dataset mode:", MODE)


## 4. Build grain samples

In [ ]:
CROP_DIR = OUTPUT_DIR / "grain_crops"
if SAVE_GRAIN_CROPS:
    CROP_DIR.mkdir(parents=True, exist_ok=True)

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def polygon_to_crop(image, coords_norm, padding=4, mask_background=True):
    """
    coords_norm: [(x1,y1), (x2,y2), ...] normalized to 0..1
    Returns a PIL crop with optional masked background.
    """
    w, h = image.size
    pts = [(x * w, y * h) for x, y in coords_norm]

    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    x1 = int(clamp(math.floor(min(xs)) - padding, 0, w - 1))
    y1 = int(clamp(math.floor(min(ys)) - padding, 0, h - 1))
    x2 = int(clamp(math.ceil(max(xs)) + padding, x1 + 1, w))
    y2 = int(clamp(math.ceil(max(ys)) + padding, y1 + 1, h))

    crop = image.crop((x1, y1, x2, y2)).convert("RGB")

    if mask_background:
        mask = Image.new("L", image.size, 0)
        draw = ImageDraw.Draw(mask)
        draw.polygon(pts, fill=255)
        mask_crop = mask.crop((x1, y1, x2, y2))
        background = Image.new("RGB", crop.size, (127, 127, 127))
        crop = Image.composite(crop, background, mask_crop)

    return crop, (x1, y1, x2, y2)


def bbox_to_crop(image, xc, yc, bw, bh, padding=4):
    w, h = image.size
    x1 = int(clamp((xc - bw / 2) * w - padding, 0, w - 1))
    y1 = int(clamp((yc - bh / 2) * h - padding, 0, h - 1))
    x2 = int(clamp((xc + bw / 2) * w + padding, x1 + 1, w))
    y2 = int(clamp((yc + bh / 2) * h + padding, y1 + 1, h))
    return image.crop((x1, y1, x2, y2)).convert("RGB"), (x1, y1, x2, y2)


samples = []

if MODE == "yolo":
    pairs = find_yolo_pairs(DATASET_DIR)
    print("YOLO image/label pairs:", len(pairs))

    instance_counter = 0
    for split, img_path, label_path in tqdm(pairs, desc="Reading YOLO dataset"):
        image = Image.open(img_path).convert("RGB")

        with open(label_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        for line_idx, line in enumerate(lines):
            parts = line.split()
            if len(parts) < 5:
                continue

            class_id = int(float(parts[0]))
            values = list(map(float, parts[1:]))

            # YOLO detection: class xc yc w h
            if len(values) == 4:
                crop, bbox = bbox_to_crop(image, *values, padding=CROP_PADDING)
                annotation_type = "bbox"

            # YOLO segmentation: class x1 y1 x2 y2 ...
            elif len(values) >= 6 and len(values) % 2 == 0:
                coords = list(zip(values[0::2], values[1::2]))
                crop, bbox = polygon_to_crop(
                    image,
                    coords,
                    padding=CROP_PADDING,
                    mask_background=MASK_BACKGROUND
                )
                annotation_type = "polygon"
            else:
                warnings.warn(f"Skipping unsupported label line: {label_path} line {line_idx+1}")
                continue

            class_name = CLASS_NAMES.get(class_id, str(class_id))
            sample_id = f"{split}_{img_path.stem}_{line_idx:04d}"

            crop_path = None
            if SAVE_GRAIN_CROPS:
                class_folder = CROP_DIR / str(class_name)
                class_folder.mkdir(parents=True, exist_ok=True)
                crop_path = class_folder / f"{sample_id}.jpg"
                crop.save(crop_path, quality=95)

            samples.append({
                "sample_id": sample_id,
                "split": split,
                "source_image": str(img_path),
                "label_path": str(label_path),
                "label_id": class_id,
                "label_name": class_name,
                "annotation_type": annotation_type,
                "bbox": bbox,
                "crop_path": str(crop_path) if crop_path else None,
                "_image": crop,
            })
            instance_counter += 1

            if MAX_SAMPLES is not None and instance_counter >= MAX_SAMPLES:
                break

        if MAX_SAMPLES is not None and instance_counter >= MAX_SAMPLES:
            break

elif MODE == "classification":
    class_dirs = [p for p in DATASET_DIR.iterdir() if p.is_dir()]
    class_dirs = sorted(class_dirs, key=lambda p: p.name.lower())
    name_to_id = {p.name: i for i, p in enumerate(class_dirs)}

    for class_dir in class_dirs:
        for img_path in class_dir.rglob("*"):
            if img_path.suffix.lower() not in IMAGE_EXTS:
                continue
            image = Image.open(img_path).convert("RGB")
            sample_id = img_path.stem

            samples.append({
                "sample_id": sample_id,
                "split": "unknown",
                "source_image": str(img_path),
                "label_path": None,
                "label_id": name_to_id[class_dir.name],
                "label_name": class_dir.name,
                "annotation_type": "classification_image",
                "bbox": None,
                "crop_path": str(img_path),
                "_image": image,
            })

            if MAX_SAMPLES is not None and len(samples) >= MAX_SAMPLES:
                break

        if MAX_SAMPLES is not None and len(samples) >= MAX_SAMPLES:
            break

print("Total grain samples:", len(samples))
print("Class distribution:", Counter(s["label_name"] for s in samples))


## 5. Quick visual check — verify the input to DINOv2

In [ ]:
if len(samples) == 0:
    raise RuntimeError("No samples were extracted. Check the dataset format and paths.")

n_show = min(12, len(samples))
chosen = random.sample(samples, n_show)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
axes = axes.flatten()

for ax, sample in zip(axes, chosen):
    ax.imshow(sample["_image"])
    ax.set_title(sample["label_name"])
    ax.axis("off")

for ax in axes[n_show:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 6. Load pretrained DINOv2

No fine-tuning is performed here. The backbone remains frozen.

The Hugging Face image processor applies the model's required resize/normalization preprocessing automatically.


In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

for p in model.parameters():
    p.requires_grad = False

hidden_size = model.config.hidden_size
print("Model:", MODEL_NAME)
print("Hidden size:", hidden_size)
print("Pooling:", POOLING)
print("All model parameters frozen:", all(not p.requires_grad for p in model.parameters()))


## 7. Extract embeddings

In [ ]:
@torch.inference_mode()
def extract_batch_embeddings(pil_images):
    inputs = processor(images=pil_images, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    outputs = model(**inputs)
    tokens = outputs.last_hidden_state

    cls_emb = tokens[:, 0, :]
    patch_mean = tokens[:, 1:, :].mean(dim=1)

    if POOLING == "cls":
        emb = cls_emb
    elif POOLING == "mean":
        emb = patch_mean
    elif POOLING == "cls_mean":
        emb = torch.cat([cls_emb, patch_mean], dim=1)
    else:
        raise ValueError("POOLING must be: cls, mean, or cls_mean")

    return emb.detach().cpu().numpy()


all_embeddings = []
metadata_rows = []

effective_batch = BATCH_SIZE if DEVICE == "cuda" else min(BATCH_SIZE, 8)

for start in tqdm(range(0, len(samples), effective_batch), desc="DINOv2 embedding extraction"):
    batch = samples[start:start + effective_batch]
    images = [s["_image"] for s in batch]

    try:
        batch_embeddings = extract_batch_embeddings(images)
    except RuntimeError as e:
        if "out of memory" in str(e).lower() and DEVICE == "cuda":
            torch.cuda.empty_cache()
            raise RuntimeError(
                "CUDA out of memory. Reduce BATCH_SIZE (for example 32 -> 16 -> 8)."
            ) from e
        raise

    all_embeddings.append(batch_embeddings)

    for s in batch:
        metadata_rows.append({
            k: v for k, v in s.items()
            if k != "_image"
        })

embeddings = np.concatenate(all_embeddings, axis=0)
metadata = pd.DataFrame(metadata_rows)

print("Embeddings shape:", embeddings.shape)
print("Metadata shape:", metadata.shape)
metadata.head()


## 8. Save embeddings and metadata

In [ ]:
np.save(OUTPUT_DIR / "dinov2_embeddings.npy", embeddings)
metadata.to_csv(OUTPUT_DIR / "dinov2_metadata.csv", index=False)

# Also save a compact NPZ for easy reuse
np.savez_compressed(
    OUTPUT_DIR / "dinov2_embeddings_and_labels.npz",
    embeddings=embeddings,
    labels=metadata["label_id"].to_numpy(),
    label_names=metadata["label_name"].astype(str).to_numpy(),
)

print("Saved:")
print(" -", OUTPUT_DIR / "dinov2_embeddings.npy")
print(" -", OUTPUT_DIR / "dinov2_metadata.csv")
print(" -", OUTPUT_DIR / "dinov2_embeddings_and_labels.npz")


## 9. PCA — first look at class separability

This does **not** train a classifier. PCA only projects the embedding space to 2D so we can visually inspect patterns.


In [ ]:
X = embeddings
y_names = metadata["label_name"].astype(str).to_numpy()

pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(StandardScaler().fit_transform(X))

plt.figure(figsize=(10, 7))
for class_name in sorted(np.unique(y_names)):
    idx = y_names == class_name
    plt.scatter(X_pca[idx, 0], X_pca[idx, 1], s=18, alpha=0.65, label=class_name)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("DINOv2 Embedding Space — PCA")
plt.legend()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total explained by 2 PCs:", pca.explained_variance_ratio_.sum())


## 10. UMAP — optional nonlinear view

In [ ]:
if RUN_UMAP:
    try:
        import umap

        reducer = umap.UMAP(
            n_components=2,
            n_neighbors=20,
            min_dist=0.1,
            metric="cosine",
            random_state=RANDOM_SEED,
        )
        X_umap = reducer.fit_transform(normalize(X))

        plt.figure(figsize=(10, 7))
        for class_name in sorted(np.unique(y_names)):
            idx = y_names == class_name
            plt.scatter(X_umap[idx, 0], X_umap[idx, 1], s=18, alpha=0.65, label=class_name)

        plt.xlabel("UMAP 1")
        plt.ylabel("UMAP 2")
        plt.title("DINOv2 Embedding Space — UMAP")
        plt.legend()
        plt.show()

    except ImportError:
        print("UMAP is not installed. Run: %pip install umap-learn")
else:
    print("RUN_UMAP = False")


## 11. Quantitative separability checks

In [ ]:
unique_classes, class_counts = np.unique(y_names, return_counts=True)

if len(unique_classes) >= 2 and np.all(class_counts >= 2):
    # Silhouette with cosine distance works well for embedding spaces.
    # For very large datasets, sample for speed.
    max_silhouette_samples = 10000
    if len(X) > max_silhouette_samples:
        idx = np.random.choice(len(X), max_silhouette_samples, replace=False)
        sil_X = normalize(X[idx])
        sil_y = y_names[idx]
    else:
        sil_X = normalize(X)
        sil_y = y_names

    score = silhouette_score(sil_X, sil_y, metric="cosine")
    print(f"Silhouette score (cosine): {score:.4f}")
    print("Interpretation: higher is better separation; near 0 means strong overlap.")
else:
    print("Need at least 2 classes with >=2 samples each for silhouette score.")

# Class-centroid cosine similarity
centroids = []
centroid_names = []

for class_name in sorted(unique_classes):
    class_emb = X[y_names == class_name]
    centroid = class_emb.mean(axis=0)
    centroids.append(centroid)
    centroid_names.append(class_name)

centroid_sim = cosine_similarity(np.stack(centroids))
centroid_df = pd.DataFrame(
    centroid_sim,
    index=centroid_names,
    columns=centroid_names
)

print("\nClass centroid cosine similarity:")
display(centroid_df.round(4))
print("\nLower off-diagonal similarity generally means classes are more distinct in embedding space.")


## 12. Feature redundancy check

In [ ]:
# Correlation across embedding dimensions
# Standardize first so dimensions are comparable.
X_scaled = StandardScaler().fit_transform(X)

corr = np.corrcoef(X_scaled, rowvar=False)
abs_corr = np.abs(corr)

# Ignore diagonal
mask = ~np.eye(abs_corr.shape[0], dtype=bool)
off_diag = abs_corr[mask]

print("Embedding dimensions:", X.shape[1])
print(f"Mean absolute feature correlation: {off_diag.mean():.4f}")
print(f"Median absolute feature correlation: {np.median(off_diag):.4f}")
print(f"Fraction of feature pairs with |corr| > 0.90: {(off_diag > 0.90).mean():.4%}")

print(
    "\nNote: some redundancy is normal in deep embeddings. "
    "If downstream ML later suffers from dimensionality/redundancy, test PCA or feature selection."
)


## 13. Robustness test — blur / brightness consistency

This checks whether the embedding of the **same grain** remains similar after a mild visual change.

High cosine similarity = DINOv2 representation is relatively stable to that change.

This is only a diagnostic; it does not prove classification performance.


In [ ]:
@torch.inference_mode()
def get_single_embedding(img):
    return extract_batch_embeddings([img])[0]

def embedding_cosine(a, b):
    return float(cosine_similarity(a.reshape(1, -1), b.reshape(1, -1))[0, 0])

n_test = min(100, len(samples))
test_samples = random.sample(samples, n_test)

records = []

for s in tqdm(test_samples, desc="Robustness test"):
    original = s["_image"]
    blurred = original.filter(ImageFilter.GaussianBlur(radius=1.5))
    darker = ImageEnhance.Brightness(original).enhance(0.8)
    brighter = ImageEnhance.Brightness(original).enhance(1.2)

    e0 = get_single_embedding(original)
    e_blur = get_single_embedding(blurred)
    e_dark = get_single_embedding(darker)
    e_bright = get_single_embedding(brighter)

    records.append({
        "label": s["label_name"],
        "blur_similarity": embedding_cosine(e0, e_blur),
        "dark_similarity": embedding_cosine(e0, e_dark),
        "bright_similarity": embedding_cosine(e0, e_bright),
    })

robustness_df = pd.DataFrame(records)

display(robustness_df.describe().loc[["mean", "std", "min", "max"]])
print("\nAverage by class:")
display(robustness_df.groupby("label").mean().round(4))


## 14. Optional: simple linear probe

Keep `RUN_LINEAR_PROBE = False` if the goal is **only** to inspect the embeddings.

If enabled, this is a quick sanity check: *can a very simple classifier use these embeddings?*  
It is not the final ML model selection stage.


In [ ]:
if RUN_LINEAR_PROBE:
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.metrics import classification_report
    from sklearn.model_selection import train_test_split

    y = metadata["label_name"].astype(str).to_numpy()

    # IMPORTANT:
    # If data comes from video/near-duplicate frames, do NOT use this random split for final evaluation.
    # Replace it with a group split by video/session/batch.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=RANDOM_SEED,
        stratify=y
    )

    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=3000)
    )
    probe.fit(X_train, y_train)
    pred = probe.predict(X_test)

    print(classification_report(y_test, pred))
else:
    print("Linear probe skipped. Set RUN_LINEAR_PROBE = True if you want a quick sanity check.")


## 15. What to look for

### DINOv2 embeddings look promising if:
- Healthy / Bad / Broken form visibly different regions in PCA/UMAP.
- The silhouette score is meaningfully above zero.
- Class centroids are less similar across different classes than within the same class.
- Mild blur/brightness changes do not completely change the embedding.
- A later simple ML model can use the embeddings without heavy feature engineering.

### If separation is weak:
Do **not** jump directly to full fine-tuning. Compare in this order:

1. Pretrained DINOv2-S vs DINOv2-B.
2. `CLS` vs `mean` vs `CLS + mean` pooling.
3. Original images vs carefully controlled color normalization.
4. Different crop/mask strategies.
5. Only then test partial fine-tuning / LoRA and compare the new embedding space against this baseline.

### Data leakage warning
If the source data comes from video, split by **video / recording session / batch**, not by random consecutive frames.


In [ ]:
# Final summary table
summary = pd.DataFrame({
    "Item": [
        "Model",
        "Fine-tuning",
        "Pooling",
        "Samples",
        "Embedding dimensions",
        "Dataset mode",
        "Output directory",
    ],
    "Value": [
        MODEL_NAME,
        "No — frozen pretrained DINOv2",
        POOLING,
        len(metadata),
        embeddings.shape[1],
        MODE,
        str(OUTPUT_DIR),
    ]
})

display(summary)
